# `numpy.einsum`: expressive contractions, measured trade-offs

`numpy.einsum` expresses array operations by naming axes.  That can make
tensor contractions compact and can avoid temporary arrays, but it is
not a general-purpose speed switch.  A dedicated NumPy operation is
often clearer and faster for a standard operation.

**Learning objectives.**  After working through this notebook, you
should be able to:

- read and write explicit `einsum` subscripts;
- translate sums, transposes, traces, matrix products, and batched
  contractions into `einsum`;
- inspect and reuse a contraction path;
- benchmark equivalent implementations without assuming that the
  shortest expression is the fastest; and
- choose a dedicated operation, broadcasting, or `einsum` based on
  clarity, memory traffic, and measurements.

**Prerequisites:** NumPy indexing, array shapes, broadcasting, and
matrix multiplication.  Allow roughly 35 minutes for the examples and
15 minutes for the exercises.  The benchmark cells are intentionally
small enough for a laptop.

## Requirements and reproducibility

This notebook uses the repository's standard environment; no packages
are installed from inside the notebook.  Timings depend on the NumPy
version, dtype, shapes, memory layout, BLAS library, thread settings,
CPU, and competing system load.  Treat the committed results as one
example and rerun them on the machine that matters to you.
To reduce thread-pool noise, the setup cell requests one thread from common
BLAS backends before importing NumPy.  This makes the teaching comparison
more repeatable, but it does not predict a multithreaded production run.


The benchmark helper reports the median and interquartile range (IQR)
of repeated calls.  Its calibration and warm-up are useful for a
teaching comparison, but this is still a microbenchmark rather than a
substitute for profiling an application.

In [1]:
import platform
import os
import statistics
import sys
import timeit

for variable in (
    "OPENBLAS_NUM_THREADS",
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "BLIS_NUM_THREADS",
):
    os.environ[variable] = "1"

import numpy as np

import pandas as pd

rng = np.random.default_rng(2026_09_23)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy:  {np.__version__}")
print(f"System: {platform.platform()}")

Python: 3.12.13
NumPy:  2.4.6
System: Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.35


Record the requested thread limit and linked BLAS implementation with
any serious performance result.

In [2]:
build_config = np.show_config(mode="dicts")
blas_config = build_config.get("Build Dependencies", {}).get("blas", {})
{
    "requested threads": {
        variable: os.environ[variable]
        for variable in (
            "OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS",
            "MKL_NUM_THREADS", "BLIS_NUM_THREADS",
        )
    },
    "BLAS": {
        key: blas_config.get(key)
        for key in ("name", "version", "detection method")
    },
}

{'requested threads': {'OPENBLAS_NUM_THREADS': '1',
  'OMP_NUM_THREADS': '1',
  'MKL_NUM_THREADS': '1',
  'BLIS_NUM_THREADS': '1'},
 'BLAS': {'name': 'blas', 'version': '3.9.0', 'detection method': 'pkgconfig'}}

## 1. Read the subscripts as axis names

Use explicit mode, with `->`, while learning:

```text
input axis labels, input axis labels -> output axis labels
```

The durable rules are:

1. Each operand gets one label per axis.
2. A label retained on the right of `->` is retained in the output.
3. A label omitted from the output is summed over.
4. A repeated label within one operand selects a diagonal.
5. The order of output labels determines the output axis order.
6. `...` stands for zero or more batch axes.

For example,

$$
c_{ij} = \sum_k a_{ik} b_{kj}
$$

becomes `"ik,kj->ij"`.  Here $i$ labels rows of $a$, $j$ labels
columns of $b$, and $k$ labels the contracted dimension.

### Reductions, inner products, and outer products

In [3]:
x = np.arange(1.0, 5.0)
y = np.arange(5.0, 9.0)

examples = {
    "sum": np.einsum("i->", x),
    "inner product": np.einsum("i,i->", x, y),
    "element-wise product": np.einsum("i,i->i", x, y),
    "outer product": np.einsum("i,j->ij", x, y),
}
examples

{'sum': np.float64(10.0),
 'inner product': np.float64(70.0),
 'element-wise product': array([ 5., 12., 21., 32.]),
 'outer product': array([[ 5.,  6.,  7.,  8.],
        [10., 12., 14., 16.],
        [15., 18., 21., 24.],
        [20., 24., 28., 32.]])}

In [4]:
assert np.allclose(examples["sum"], x.sum())
assert np.allclose(examples["inner product"], np.dot(x, y))
assert np.allclose(examples["element-wise product"], x * y)
assert np.allclose(examples["outer product"], np.outer(x, y))

### Axis permutation, diagonal, and trace

In [5]:
matrix = np.arange(12).reshape(3, 4)
square = np.arange(16).reshape(4, 4)

transposed = np.einsum("ij->ji", matrix)
diagonal = np.einsum("ii->i", square)
trace = np.einsum("ii->", square)

assert np.array_equal(transposed, matrix.T)
assert np.array_equal(diagonal, np.diag(square))
assert trace == np.trace(square)

transposed, diagonal, trace

(array([[ 0,  4,  8],
        [ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11]]),
 array([ 0,  5, 10, 15]),
 np.int64(30))

These examples demonstrate the notation, but `matrix.T`,
`np.diagonal`, and `np.trace` communicate the standard operations more
directly.  Do not replace familiar operations merely to use `einsum`.

### Matrix and batched matrix multiplication

In [6]:
a = rng.normal(size=(3, 4))
b = rng.normal(size=(4, 2))
product = np.einsum("ik,kj->ij", a, b)
assert np.allclose(product, a @ b)

batch_a = rng.normal(size=(5, 3, 4))
batch_b = rng.normal(size=(5, 4, 2))
batch_product = np.einsum("...ik,...kj->...ij", batch_a, batch_b)
assert np.allclose(batch_product, batch_a @ batch_b)

product.shape, batch_product.shape

((3, 2), (5, 3, 2))

### A contraction for which `einsum` is especially readable

Suppose each row $x_b$ of `samples` is a state vector and `operator`
contains a bilinear form.  The quadratic form for batch member $b$ is

$$
q_b = \sum_i \sum_j X_{bi} A_{ij} X_{bj},
$$

where $b$ labels a batch member and $i,j$ label vector components.
All three operands and both contractions appear in one expression.

In [7]:
samples = rng.normal(size=(6, 4))
operator = rng.normal(size=(4, 4))

quadratic = np.einsum("bi,ij,bj->b", samples, operator, samples)
reference = np.array([row @ operator @ row for row in samples])

assert np.allclose(quadratic, reference)
quadratic

array([-2.91609562, -5.47360897, -3.11144922, -4.09074823, -0.92559929,
        0.70515615])

### A correctness trap: complex conjugation is not implicit

Repeating a subscript means multiply and sum; it does **not** conjugate
a complex operand.  Use `.conj()` explicitly, or a dedicated operation
such as `np.vdot` when its semantics match the problem.

In [8]:
z = np.array([1 + 2j, 3 - 1j])

bilinear = np.einsum("i,i->", z, z)
hermitian_inner = np.einsum("i,i->", z.conj(), z)

assert np.allclose(hermitian_inner, np.vdot(z, z))
{"without conjugation": bilinear, "with conjugation": hermitian_inner}

{'without conjugation': np.complex128(5-2j),
 'with conjugation': np.complex128(15+0j)}

## 2. Contraction order can change the algorithm

Consider `"ab,bc,cd->ad"`.  Evaluating all three operands as one
direct sum can perform far more arithmetic than first forming a small
intermediate.  `np.einsum_path` estimates candidate contraction orders.

`np.einsum` defaults to `optimize=False`, whereas
`np.einsum_path` defaults to a greedy search.  Therefore a readable
multi-operand expression is not automatically evaluated in a good
order: pass an optimization policy or a precomputed path deliberately.

In [9]:
chain_a = rng.normal(size=(100, 10))
chain_b = rng.normal(size=(10, 1_000))
chain_c = rng.normal(size=(1_000, 5))

chain_path, path_report = np.einsum_path(
    "ab,bc,cd->ad", chain_a, chain_b, chain_c, optimize="greedy"
)
print(path_report)

  Complete contraction:  ab,bc,cd->ad
         Naive scaling:  4
     Optimized scaling:  3
      Naive FLOP count:  1.500e+07
  Optimized FLOP count:  1.100e+05
   Theoretical speedup:  136.362
  Largest intermediate:  5.000e+02 elements
--------------------------------------------------------------------------
scaling                  current                                remaining
--------------------------------------------------------------------------
   3                   cd,bc->db                                ab,db->ad
   3                   db,ab->ad                                   ad->ad


In [10]:
unoptimized = np.einsum(
    "ab,bc,cd->ad", chain_a, chain_b, chain_c, optimize=False
)
optimized = np.einsum(
    "ab,bc,cd->ad", chain_a, chain_b, chain_c, optimize=chain_path
)
assert np.allclose(unoptimized, optimized)

The path search itself costs time.  Reuse the returned `chain_path`
when the expression and shapes recur.  The `"optimal"` search explores
combinations and scales exponentially with the number of operands;
`"greedy"` is the pragmatic default for most larger expressions.

## 3. Benchmark equivalent work, not syntax

In [11]:
def benchmark(cases, *, repeat=5, target_time=0.05):
    '''Benchmark zero-argument callables after warm-up and calibration.'''
    rows = []
    for label, function in cases.items():
        function()  # warm caches and lazy runtime initialization
        timer = timeit.Timer(function)
        number = 1
        elapsed = timer.timeit(number=number)

        while elapsed < target_time and number < 1_000_000:
            multiplier = max(
                2,
                min(10, int(np.ceil(target_time / max(elapsed, 1e-12)))),
            )
            number = min(1_000_000, number * multiplier)
            elapsed = timer.timeit(number=number)

        samples = np.array(timer.repeat(repeat=repeat, number=number)) / number
        rows.append(
            {
                "implementation": label,
                "median (us)": 1e6 * statistics.median(samples),
                "IQR (us)": 1e6 * (
                    np.percentile(samples, 75) - np.percentile(samples, 25)
                ),
                "calls per repeat": number,
            }
        )

    result = pd.DataFrame(rows)
    result["relative to fastest"] = (
        result["median (us)"] / result["median (us)"].min()
    )
    return result.set_index("implementation").round(2)

The first implementation below is not assumed to be the baseline;
`relative to fastest` is computed from the fastest median in that run.
All comparisons first check numerical equivalence.

### Counterexample 1: matrix multiplication already has a specialist

For two ordinary matrices, `@` states the intent and can use optimized
BLAS routines.  An unoptimized `einsum` may use a much less efficient
loop, while path optimization also has overhead.  Measure rather than
assuming `optimize="greedy"` will beat `@`.

In [12]:
n = 256
left = rng.normal(size=(n, n))
right = rng.normal(size=(n, n))

expected = left @ right
assert np.allclose(
    np.einsum("ik,kj->ij", left, right, optimize=False), expected
)
assert np.allclose(
    np.einsum("ik,kj->ij", left, right, optimize="greedy"), expected
)

benchmark(
    {
        "left @ right": lambda: left @ right,
        "einsum, optimize=False": lambda: np.einsum(
            "ik,kj->ij", left, right, optimize=False
        ),
        "einsum, optimize='greedy'": lambda: np.einsum(
            "ik,kj->ij", left, right, optimize="greedy"
        ),
    }
)

,median (us),IQR (us),calls per repeat,relative to fastest
implementation,,,,
left @ right,696.79,14.92,80,1.00
"einsum, optimize=False",4294.31,361.01,20,6.16
"einsum, optimize='greedy'",716.38,16.62,80,1.03


### Counterexample 2: path planning dominates tiny work

For length-eight vectors, the arithmetic is trivial.  Asking for a
greedy path on every call can cost more than the contraction.  A
dedicated `np.dot` is also more obvious to most readers.

In [13]:
tiny_x = rng.normal(size=8)
tiny_y = rng.normal(size=8)
expected = np.dot(tiny_x, tiny_y)

assert np.allclose(np.einsum("i,i->", tiny_x, tiny_y), expected)

benchmark(
    {
        "np.dot": lambda: np.dot(tiny_x, tiny_y),
        "einsum, optimize=False": lambda: np.einsum(
            "i,i->", tiny_x, tiny_y, optimize=False
        ),
        "einsum, optimize='greedy'": lambda: np.einsum(
            "i,i->", tiny_x, tiny_y, optimize="greedy"
        ),
    },
    target_time=0.03,
)

,median (us),IQR (us),calls per repeat,relative to fastest
implementation,,,,
np.dot,0.64,0.01,60000,1.00
"einsum, optimize=False",1.45,0.02,30000,2.26
"einsum, optimize='greedy'",10.19,1.15,3000,15.83


### A case where fusion can help

`(values * weights).sum(axis=1)` creates a full temporary product.
`"bf,bf->b"` can multiply and reduce in one contraction.  Here $b$
labels observations and $f$ labels features.  Fewer allocated bytes can
improve performance, especially once the arrays exceed cache capacity.
The result is still empirical: strides, dtype, and hardware can reverse
the ranking.

In [14]:
values = rng.normal(size=(2_000, 512))
weights = rng.normal(size=(2_000, 512))

with_temporary = (values * weights).sum(axis=1)
fused = np.einsum("bf,bf->b", values, weights, optimize=False)
assert np.allclose(fused, with_temporary)

benchmark(
    {
        "multiply, then sum": lambda: (values * weights).sum(axis=1),
        "einsum fused contraction": lambda: np.einsum(
            "bf,bf->b", values, weights, optimize=False
        ),
    }
)

,median (us),IQR (us),calls per repeat,relative to fastest
implementation,,,,
"multiply, then sum",2244.27,244.66,30,3.31
einsum fused contraction,678.82,74.56,80,1.00


### Multi-operand contractions: compare with `multi_dot`

For a chain of two-dimensional matrix products,
`np.linalg.multi_dot` already chooses a multiplication order and states
the intent clearly.  `einsum` becomes attractive when the contraction
is not a conventional matrix chain.  Notice how severely the default
unoptimized contraction can degrade performance for these shapes.

In [15]:
expected = np.linalg.multi_dot([chain_a, chain_b, chain_c])
assert np.allclose(optimized, expected)

benchmark(
    {
        "np.linalg.multi_dot": lambda: np.linalg.multi_dot(
            [chain_a, chain_b, chain_c]
        ),
        "einsum, optimize=False": lambda: np.einsum(
            "ab,bc,cd->ad",
            chain_a,
            chain_b,
            chain_c,
            optimize=False,
        ),
        "einsum, cached greedy path": lambda: np.einsum(
            "ab,bc,cd->ad",
            chain_a,
            chain_b,
            chain_c,
            optimize=chain_path,
        ),
    }
)

,median (us),IQR (us),calls per repeat,relative to fastest
implementation,,,,
np.linalg.multi_dot,6.70,0.16,8000,1.00
"einsum, optimize=False",8439.05,94.53,6,1260.31
"einsum, cached greedy path",22.83,1.17,3000,3.41


## 4. A practical decision guide

Prefer a dedicated operation when it expresses the computation
directly: `sum`, `mean`, `trace`, `.T`, `@`, `np.dot`, or
`np.linalg.multi_dot`.  Advantages include familiar semantics, easier
review, and access to specialized implementations.  The disadvantage is
that a sequence of operations can allocate intermediates or obscure a
non-standard contraction.

Consider `einsum` when named axes make a multi-axis contraction clearer,
when it fuses a multiply-and-reduce operation, or when it replaces
several error-prone reshapes and transposes.  Its disadvantages are a
compact notation unfamiliar to some readers, easy index-label mistakes,
non-obvious complex-conjugation semantics, contraction-path decisions,
and performance that is not predictable from the expression alone.

For performance-sensitive code:

1. write a clear reference implementation;
2. assert numerical equivalence with realistic shapes and dtypes;
3. inspect `np.einsum_path` for three or more operands;
4. include path-planning cost if production code pays it, otherwise
   precompute and benchmark the reused path;
5. benchmark in the real application environment; and
6. profile peak memory as well as runtime when temporaries are large.

## 5. Exercises

### Exercise 1 — translate a batched weighted sum

`measurements` has shape `(experiment, time, sensor)` and
`calibration` has shape `(sensor,)`.  Compute

$$
y_{et} = \sum_s M_{ets} c_s,
$$

where $e$ labels experiments, $t$ time points, and $s$ sensors.

1. Write an explicit `einsum` expression.
2. Write an equivalent expression using a dedicated operation.
3. Predict which is clearer and which is faster before measuring.

Success criterion: both results have shape `(7, 20)` and agree within
floating-point tolerance.  Timebox: 5 minutes.

In [16]:
measurements = rng.normal(size=(7, 20, 12))
calibration = rng.normal(size=12)

# Write both implementations here before opening the solution.

<details>
<summary>Complete solution to Exercise 1</summary>

`np.einsum("ets,s->et", measurements, calibration)` directly mirrors
the mathematical labels.  `measurements @ calibration` is the
equivalent dedicated matrix-vector operation over the final axis.  The
latter is shorter and familiar, so it is a strong default here.  The
performance prediction must be checked locally.
</details>

In [17]:
exercise_1_einsum = np.einsum(
    "ets,s->et", measurements, calibration, optimize=False
)
exercise_1_matmul = measurements @ calibration

assert exercise_1_einsum.shape == (7, 20)
assert np.allclose(exercise_1_einsum, exercise_1_matmul)

### Exercise 2 — diagnose the missing conjugate

`states` has shape `(batch, component)` and complex dtype.  A proposed
implementation of the squared norm is
`np.einsum("bi,bi->b", states, states)`.  Predict whether it is correct,
construct a two-element counterexample, and repair it.

Success criterion: the repaired result is real and non-negative up to
floating-point round-off, and it agrees with a dedicated NumPy
operation.  Timebox: 5 minutes.

In [18]:
states = rng.normal(size=(5, 4)) + 1j * rng.normal(size=(5, 4))

# Diagnose and repair the proposed contraction here.

<details>
<summary>Complete solution to Exercise 2</summary>

The proposal computes a bilinear form, not a Hermitian norm, because
`einsum` does not conjugate automatically.  Conjugate one operand:
`np.einsum("bi,bi->b", states.conj(), states)`.  The dedicated
`np.linalg.vector_norm(states, axis=1) ** 2` states the norm operation
more directly.
</details>

In [19]:
proposed = np.einsum("bi,bi->b", states, states)
repaired = np.einsum("bi,bi->b", states.conj(), states)
norm_squared = np.linalg.vector_norm(states, axis=1) ** 2

assert not np.allclose(proposed, norm_squared)
assert np.allclose(repaired, norm_squared)
assert np.all(repaired.real >= -10 * np.finfo(float).eps)
repaired

array([ 3.68316656+0.j,  8.56967496+0.j,  4.16789371+0.j, 10.64180756+0.j,
        1.81203991+0.j])

## References

- [NumPy: `einsum`](https://numpy.org/doc/stable/reference/generated/numpy.einsum.html)
- [NumPy: `einsum_path`](https://numpy.org/doc/stable/reference/generated/numpy.einsum_path.html)
- [NumPy: `matmul`](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html)
- [NumPy: `linalg.multi_dot`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.multi_dot.html)

The API details in this notebook were checked against the official NumPy
documentation.  Recheck version-specific behavior when maintaining the
material.